In [ ]:
import potcorr
import numpy as np
import lab
import matplotlib.pyplot as plt
import io_xsf
import psutil
import os

In [ ]:
cell=lab.mos2_12x12

ttkw=potcorr.PotCorr(cell)
ttkw.fft_init()
ttkw.get_vcoul()
ttkw.read_rho_tot(ttkw.folder+'drho.xsf')

ttkw.read_chi(cell['folder']+'../chi_12x12/chimat.h5',cell['folder']+'../chi_12x12/chi0mat.h5' )
ttkw.epsmat_init()
#print(mos2.q_ind_tt)
ttkw.get_epsmat(G_ind_cut = 1000, kernel='3d')
#ttkw.epsmat_inv(G_ind_cut = 1000)
del ttkw.Chi0
del ttkw.Chi1
ttkw.rho2pot_tot(kernel='3d')
ttkw.pot_tot2bare(kernel='3d', G_ind_cut=1000)

#print(mos2.pot_tot_r)
ttkw.pot2rho_bare(kernel='3d', ncharge=1)

#ttkw.write_xsf(ftype='rho_bare', filedir=ttkw.folder+'rho_bare.xsf')
#mos2.write_xsf(ftype='rho_bare', filedir=mos2.folder+'rho_bare.xsf')
info = psutil.virtual_memory()
print(u'Occupied memory：',psutil.Process(os.getpid()).memory_info().rss/1024/1024, 'MB')
print(u'Total memory：',info.total/1024/1024, 'MB')
print(u'Percent：',info.percent,'%')
print(u'Number of Cup：',psutil.cpu_count())

In [ ]:
rho0 = ttkw.rho_bare_r.real

In [ ]:
new_shape = (720, 720,225)
rho_new = np.zeros(new_shape)
rho_new[180:540, 180:540, :] = rho0

In [ ]:
np.save(cell['folder']+'rho_bare_12-24.npy',rho_new)

In [ ]:
cell=lab.mos2_12to24
ttkw=potcorr.PotCorr(cell)
ttkw.fft_init()
ttkw.get_vcoul()

In [ ]:
ttkw.rho_bare_r = np.load(cell['folder']+'rho_bare_12-24.npy')
ttkw.rho_bare_k = np.fft.fftn(ttkw.rho_bare_r)
ttkw.rho2pot_bare(kernel='3d')

In [ ]:
#ttkw.read_chi(cell['folder']+'../chi_24x24/chimat.h5',cell['folder']+'../chi_24x24/chi0mat.h5' )
ttkw.read_chi(cell['folder']+'../chi_24x24_fermi_-0.15/chimat.h5',cell['folder']+'../chi_24x24_fermi_-0.15/chi0mat.h5' )
ttkw.epsmat_init()
#print(mos2.q_ind_tt)
ttkw.get_epsmat(G_ind_cut = 600, kernel='3d')
ttkw.epsmat_inv(G_ind_cut = 600)
del ttkw.Chi0
del ttkw.Chi1

In [ ]:
ttkw.pot_bare2tot(kernel='3d', G_ind_cut=100)

In [ ]:
info = psutil.virtual_memory()
print(u'Occupied memory：',psutil.Process(os.getpid()).memory_info().rss/1024/1024, 'MB')
print(u'Total memory：',info.total/1024/1024, 'MB')
print(u'Percent：',info.percent,'%')
print(u'Number of Cup：',psutil.cpu_count())

In [ ]:
#pot_model1 = np.load(cell['folder']+'pot_tot_12-24_fermi_-0.15_G200.npy')
#pot_model2 = np.load(cell['folder']+'pot_tot_12-24_fermi_-0.15_G400.npy')
#pot_model3 = np.load(cell['folder']+'pot_tot_12-24_fermi_-0.15_G600.npy')
pot_model4 = np.load(cell['folder']+'pot_tot_12-24_fermi_-0.15_G800.npy')

In [ ]:
pot_model5 = np.load(cell['folder']+'pot_tot_12-24_fermi_-0.25_G800.npy')

In [ ]:
ttkw.lattpara

In [ ]:
import utli
nx, ny, nz = ttkw.fft_nx, ttkw.fft_ny, ttkw.fft_nz
potential_tot1 = pot_model1[:,:,int(0.45*nz):int(0.55*nz)]#/1.01213
potential_tot2 = pot_model2[:,:,int(0.45*nz):int(0.55*nz)]#/1.01213
potential_tot3 = pot_model3[:,:,int(0.45*nz):int(0.55*nz)]#/1.01213
potential_tot4 = pot_model4[:,:,int(0.45*nz):int(0.55*nz)]#/1.01213
A = float(ttkw.lattpara[0])
C = 0.2*float(ttkw.lattpara[2])

print(A)
print(nz)


dist_arr_tot = utli.distance_array(potential_tot.shape[0], potential_tot.shape[1], potential_tot.shape[2],A,A,C, defect_loc='center')
#dist_arr_dv = utli.distance_array(potential_dv.shape[0], potential_dv.shape[1], potential_dv.shape[2], A,A,C,defect_loc='center')

#dist_arr_tot = utli.distance_array(potential_tot.shape[0], potential_tot.shape[1], potential_tot.shape[2],A,A,C, defect_loc='center')
#dist_arr_dv = utli.distance_array(potential_dv.shape[0], potential_dv.shape[1], potential_dv.shape[2], A,A,C,defect_loc='center')

fig, ax = plt.subplots(2,2,figsize=(12,12),dpi=100)

ax[0,0].scatter(dist_arr_tot.flatten(), potential_tot1.flatten(), label='model G200')
ax[0,1].scatter(dist_arr_tot.flatten(), potential_tot2.flatten(), label='model G400')
ax[1,0].scatter(dist_arr_tot.flatten(), potential_tot3.flatten(), label='model G600')
ax[1,1].scatter(dist_arr_tot.flatten(), potential_tot4.flatten(), label='model G800')
#plt.scatter(dist_arr_tot.flatten(), potential_tot.flatten()*1.05, label='model with interp')
#plt.scatter(dist_arr_dv.flatten(), potential_dv.flatten(), label='dft')
#plt.scatter(dist_arr_tot[:,:,11].flatten(), potential_tot[:,:,11].flatten(), label='model upper S plane')
#plt.scatter(dist_arr_dv[:,:,17].flatten(), potential_dv[:,:,17].flatten(), label='dft upper S plane')
#plt.scatter(dist_arr_tot[:,:,6].flatten(), potential_tot[:,:,6].flatten(), label='model Mo plane')
#plt.scatter(dist_arr_dv[:,:,9].flatten(), potential_dv[:,:,9].flatten(), label='dft Mo plane')
#plt.scatter(dist_arr_tot[:,:,0].flatten(), potential_tot[:,:,0].flatten(), label='model lower S plane')
#plt.scatter(dist_arr_dv[:,:,0].flatten(), potential_dv[:,:,0].flatten(), label='dft lower S plane')

plt.xlabel('Distance from Defect')
plt.ylabel('Potential')
plt.legend()
plt.show()

In [ ]:
import utli
nx, ny, nz = ttkw.fft_nx, ttkw.fft_ny, ttkw.fft_nz
potential_tot5 = pot_model5[:,:,int(0.45*nz):int(0.55*nz)]#/1.01213
potential_tot4 = pot_model4[:,:,int(0.45*nz):int(0.55*nz)]
A = float(ttkw.lattpara[0])
C = 0.2*float(ttkw.lattpara[2])

print(A)
print(nz)


dist_arr_tot = utli.distance_array(potential_tot5.shape[0], potential_tot5.shape[1], potential_tot5.shape[2],A,A,C, defect_loc='center')
#dist_arr_dv = utli.distance_array(potential_dv.shape[0], potential_dv.shape[1], potential_dv.shape[2], A,A,C,defect_loc='center')

#dist_arr_tot = utli.distance_array(potential_tot.shape[0], potential_tot.shape[1], potential_tot.shape[2],A,A,C, defect_loc='center')
#dist_arr_dv = utli.distance_array(potential_dv.shape[0], potential_dv.shape[1], potential_dv.shape[2], A,A,C,defect_loc='center')

fig, ax = plt.subplots(figsize=(8,6),dpi=100)

ax.scatter(dist_arr_tot.flatten(), potential_tot5.flatten(), label='fermi -0.25')
ax.scatter(dist_arr_tot.flatten(), potential_tot4.flatten(), label='fermi -0.15')
#plt.scatter(dist_arr_tot.flatten(), potential_tot.flatten()*1.05, label='model with interp')
#plt.scatter(dist_arr_dv.flatten(), potential_dv.flatten(), label='dft')
#plt.scatter(dist_arr_tot[:,:,11].flatten(), potential_tot[:,:,11].flatten(), label='model upper S plane')
#plt.scatter(dist_arr_dv[:,:,17].flatten(), potential_dv[:,:,17].flatten(), label='dft upper S plane')
#plt.scatter(dist_arr_tot[:,:,6].flatten(), potential_tot[:,:,6].flatten(), label='model Mo plane')
#plt.scatter(dist_arr_dv[:,:,9].flatten(), potential_dv[:,:,9].flatten(), label='dft Mo plane')
#plt.scatter(dist_arr_tot[:,:,0].flatten(), potential_tot[:,:,0].flatten(), label='model lower S plane')
#plt.scatter(dist_arr_dv[:,:,0].flatten(), potential_dv[:,:,0].flatten(), label='dft lower S plane')

plt.xlabel('Distance from Defect')
plt.ylabel('Potential')
plt.legend()
plt.show()

In [ ]:
plt.plot(range(nz), np.sum(pot_model4, axis=(0,1)))
plt.plot(range(nz), np.sum(pot_model5, axis=(0,1)))

In [ ]:
fig, ax = plt.subplots(figsize=(6,6),dpi=100)

ax.scatter(dist_arr_tot.flatten(), potential_tot1.flatten(), label='model G200')
ax.scatter(dist_arr_tot.flatten(), potential_tot2.flatten(), label='model G400')
ax.scatter(dist_arr_tot.flatten(), potential_tot3.flatten(), label='model G600')
ax.scatter(dist_arr_tot.flatten(), potential_tot4.flatten(), label='model G800')
#plt.scatter(dist_arr_tot.flatten(), potential_tot.flatten()*1.05, label='model with interp')
#plt.scatter(dist_arr_dv.flatten(), potential_dv.flatten(), label='dft')
#plt.scatter(dist_arr_tot[:,:,11].flatten(), potential_tot[:,:,11].flatten(), label='model upper S plane')
#plt.scatter(dist_arr_dv[:,:,17].flatten(), potential_dv[:,:,17].flatten(), label='dft upper S plane')
#plt.scatter(dist_arr_tot[:,:,6].flatten(), potential_tot[:,:,6].flatten(), label='model Mo plane')
#plt.scatter(dist_arr_dv[:,:,9].flatten(), potential_dv[:,:,9].flatten(), label='dft Mo plane')
#plt.scatter(dist_arr_tot[:,:,0].flatten(), potential_tot[:,:,0].flatten(), label='model lower S plane')
#plt.scatter(dist_arr_dv[:,:,0].flatten(), potential_dv[:,:,0].flatten(), label='dft lower S plane')

plt.xlabel('Distance from Defect')
plt.ylabel('Potential')
plt.legend()
plt.show()

In [ ]:
#plt.scatter(dist_arr_tot.flatten(), potential_tot.flatten(), label='model')
#plt.scatter(dist_arr_tot.flatten(), potential_tot.flatten()*1.05, label='model with interp')
#plt.scatter(dist_arr_dv.flatten(), potential_dv.flatten(), label='dft')
plt.scatter(dist_arr_tot[:,:,21].flatten(), potential_tot[:,:,21].flatten(), label='model upper S plane')
#plt.scatter(dist_arr_dv[:,:,17].flatten(), potential_dv[:,:,17].flatten(), label='dft upper S plane')
#plt.scatter(dist_arr_tot[:,:,10].flatten(), potential_tot[:,:,10].flatten(), label='model Mo plane')
#plt.scatter(dist_arr_dv[:,:,9].flatten(), potential_dv[:,:,9].flatten(), label='dft Mo plane')
plt.scatter(dist_arr_tot[:,:,0].flatten(), potential_tot[:,:,0].flatten(), label='model lower S plane')
#plt.scatter(dist_arr_dv[:,:,0].flatten(), potential_dv[:,:,0].flatten(), label='dft lower S plane')

plt.xlabel('Distance from Defect')
plt.ylabel('Potential')
plt.legend()
plt.show()




In [ ]:
plt.plot(range(225),pot_model4[ttkw.fft_nx//2, ttkw.fft_ny//2, :] )
plt.plot(range(225),pot_model5[ttkw.fft_nx//2, ttkw.fft_ny//2, :] )

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
z_plot1 = pot_model1[:,:,ttkw.fft_nz//2]
z_plot2 = pot_model2[:,:,ttkw.fft_nz//2]
z_plot3 = pot_model3[:,:,ttkw.fft_nz//2]
z_plot4 = pot_model4[:,:,ttkw.fft_nz//2]

fig, ax = plt.subplots(2,2,figsize=(12,12),dpi=100)
hb1 = ax[0,0].scatter(x_plot, y_plot, c=z_plot1, 
                #gridsize=100,
    #            vmax=0.2,
    #            vmin=0.0,
                    cmap='coolwarm')
ax[0,0].set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax[0,0],fraction = 0.035)

hb2 = ax[0,1].scatter(x_plot, y_plot, c=z_plot2, 
                #gridsize=100,
    #            vmax=0.2,
    #            vmin=0.0,
                    cmap='coolwarm')
ax[0,1].set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb2 = fig.colorbar(hb2, ax=ax[0,1],fraction = 0.035)

hb3 = ax[1,0].scatter(x_plot, y_plot, c=z_plot3, 
                #gridsize=100,
    #            vmax=0.2,
    #            vmin=0.0,
                    cmap='coolwarm')
ax[1,0].set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb3 = fig.colorbar(hb3, ax=ax[1,0],fraction = 0.035)

hb4 = ax[1,1].scatter(x_plot, y_plot, c=z_plot4, 
                #gridsize=100,
    #            vmax=0.2,
    #            vmin=0.0,
                    cmap='coolwarm')
ax[1,1].set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb4 = fig.colorbar(hb4, ax=ax[1,1],fraction = 0.035)

In [ ]:
ttkw.pot_tot_r = pot_model4
ttkw.pot_tot_k = np.fft.fftn(ttkw.pot_tot_r)

In [ ]:
ttkw.pot2rho_tot(kernel='3d', ncharge=0)

In [ ]:

x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
hb1 = ax.scatter(x_plot, y_plot, c=ttkw.rho_tot_r.real[:,:,ttkw.fft_nz//2], 
                #gridsize=100,
    #            vmax=0.2,
    #            vmin=0.0,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)

In [ ]:
#plt.plot(range(225),np.sum(ttkw.rho_tot_r.real[:,:,:], axis=(0,1) ))
plt.plot(range(720),np.sum(ttkw.rho_tot_r.real[:,:,:], axis=(2,1) ))
#plt.plot(range(225),np.sum(pot_model4[:,:,:], axis=(0,1) ))

In [ ]:
chgden_tot = ttkw.rho_tot_r.real[:,:,int(0.45*nz):int(0.55*nz)]
plt.scatter(dist_arr_tot.flatten(), chgden_tot.flatten(), label='model upper S plane')
#plt.scatter(dist_arr_dv[:,:,17].flatten(), potential_dv[:,:,17].flatten(), label='dft upper S plane')
#plt.scatter(dist_arr_tot[:,:,10].flatten(), potential_tot[:,:,10].flatten(), label='model Mo plane')
#plt.scatter(dist_arr_dv[:,:,9].flatten(), potential_dv[:,:,9].flatten(), label='dft Mo plane')
#plt.scatter(dist_arr_tot.flatten(), chgden_tot.flatten(), label='model lower S plane')
#plt.scatter(dist_arr_dv[:,:,0].flatten(), potential_dv[:,:,0].flatten(), label='dft lower S plane')

plt.xlabel('Distance from Defect')
plt.ylabel('Charge density')
#plt.legend()
plt.show()

In [ ]:
ttkw.write_xsf(ftype='rho_tot', filedir=cell['folder']+'rho_scr_12-24_fermi_-0.25.xsf')

In [ ]:
np.save(cell['folder']+'pot_tot_12-24.npy',pot_model)

In [ ]:
ttkw.get_vcoul_sr(alpha=1.5)
pot_sr = np.fft.ifftn(ttkw.v_coul_sr*np.fft.fftn(ttkw.rho_tot_r)).real

cmap = plt.cm.get_cmap('coolwarm', 70)
fig, ax = plt.subplots(figsize=(16,6),dpi=100)
#ax.plot(range(len(ttkw.pot_bare_r)),ttkw.pot_bare_r[ttkw.fft_nx//2, ttkw.fft_ny//2, :] )
#ax.plot(range(len(pot_iso)), pot_iso[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
#ax.plot(range(len(pot_iso)), pot_iso_c[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
for i in range(140):
    ax.plot(range(ttkw.fft_nx), pot_sr[:, ttkw.fft_ny//2, ttkw.fft_nz//2+70-i], color=cmap(np.abs(i-70)))
plt.show()

In [ ]:
coarse_data = ttkw.rho_bare_r
f_transform = np.fft.fftn(coarse_data)

# 获取原始数据的维度
nx, ny, nz = f_transform.shape

# 创建一个用0填充的更大的傅里叶空间数组
new_f_transform = np.zeros((720,720, 450), dtype=complex)

# 将原始的傅里叶变换数据复制到新的傅里叶空间中心
new_f_transform[:nx//2, :ny//2, :nz//2] = f_transform[:nx//2, :ny//2, :nz//2]
new_f_transform[-nx//2:, :ny//2, :nz//2] = f_transform[-nx//2:, :ny//2, :nz//2]
new_f_transform[:nx//2, -ny//2:, :nz//2] = f_transform[:nx//2, -ny//2:, :nz//2]
new_f_transform[-nx//2:, -ny//2:, :nz//2] = f_transform[-nx//2:, -ny//2:, :nz//2]
new_f_transform[:nx//2, :ny//2, -nz//2:] = f_transform[:nx//2, :ny//2, -nz//2:]
new_f_transform[-nx//2:, :ny//2, -nz//2:] = f_transform[-nx//2:, :ny//2, -nz//2:]
new_f_transform[:nx//2, -ny//2:, -nz//2:] = f_transform[:nx//2, -ny//2:, -nz//2:]
new_f_transform[-nx//2:, -ny//2:, -nz//2:] = f_transform[-nx//2:, -ny//2:, -nz//2:]

# 执行逆傅里叶变换并缩放（因为数组大小已经改变）
fine_data2 = np.fft.ifftn(new_f_transform).real * 8

In [ ]:
cmap = plt.cm.get_cmap('hot', 50)
fig, ax = plt.subplots(figsize=(16,6),dpi=100)
#ax.plot(range(len(ttkw.pot_bare_r)),ttkw.pot_bare_r[ttkw.fft_nx//2, ttkw.fft_ny//2, :] )
#ax.plot(range(len(pot_iso)), pot_iso[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
#ax.plot(range(len(pot_iso)), pot_iso_c[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
for i in range(100):
    ax.plot(range(ttkw.fft_nx), ttkw.rho_tot_r[:, ttkw.fft_ny//2, ttkw.fft_nz//2-50+i], color=cmap(np.abs(i-50)))
plt.show()

In [ ]:
cmap = plt.cm.get_cmap('coolwarm', 100)
fig, ax = plt.subplots(figsize=(16,6),dpi=100)
#ax.plot(range(len(ttkw.pot_bare_r)),ttkw.pot_bare_r[ttkw.fft_nx//2, ttkw.fft_ny//2, :] )
#ax.plot(range(len(pot_iso)), pot_iso[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
#ax.plot(range(len(pot_iso)), pot_iso_c[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
for i in range(200):
    ax.plot(range(720), fine_data2[:, 720//2, 450//2-100+i], color=cmap(np.abs(i-100)))
plt.ylim(-0.0015, 0.006)
plt.show()

In [ ]:
cmap = plt.cm.get_cmap('coolwarm', 100)
fig, ax = plt.subplots(figsize=(16,6),dpi=100)
#ax.plot(range(len(ttkw.pot_bare_r)),ttkw.pot_bare_r[ttkw.fft_nx//2, ttkw.fft_ny//2, :] )
#ax.plot(range(len(pot_iso)), pot_iso[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
#ax.plot(range(len(pot_iso)), pot_iso_c[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
for i in range(200):
    ax.plot(range(720), fine_data[:, 720//2, 450//2-100+i], color=cmap(np.abs(i-100)))
plt.ylim(-0.0015, 0.006)
plt.show()

In [ ]:
cmap = plt.cm.get_cmap('hot', 50)
fig, ax = plt.subplots(figsize=(16,6),dpi=100)
#ax.plot(range(len(ttkw.pot_bare_r)),ttkw.pot_bare_r[ttkw.fft_nx//2, ttkw.fft_ny//2, :] )
#ax.plot(range(len(pot_iso)), pot_iso[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
#ax.plot(range(len(pot_iso)), pot_iso_c[ttkw.fft_nx//2, ttkw.fft_ny//2, :])
for i in range(100):
    ax.plot(range(ttkw.fft_nx), ttkw.rho_bare_r[:, ttkw.fft_ny//2, ttkw.fft_nz//2-50+i].real, color=cmap(np.abs(i-50)))
plt.show()